In [1]:
import geopandas as gpd

# ==========================================
# 1. Configuración de rutas
# ==========================================
# Reemplaza estas rutas con la ubicación real de tus archivos
archivo_entrada = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\edificios_regularizados.gpkg"
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
#"ruta/a/tu/capa_centroides.gpkg"

# Si tu GeoPackage tiene múltiples capas y no es la primera, especifica el nombre:
# nombre_capa = "mi_capa_especifica" 

try:
    # ==========================================
    # 2. Cargar los datos
    # ==========================================
    print("Leyendo la capa de polígonos...")
    # Si necesitas una capa específica, añade: layer=nombre_capa dentro de read_file
    gdf_poligonos = gpd.read_file(archivo_entrada)
    
    # ==========================================
    # 3. Calcular los centroides
    # ==========================================
    print("Calculando los centroides...")
    # Creamos una copia para mantener todos los datos (atributos) originales
    gdf_centroides = gdf_poligonos.copy()
    
    # Reemplazamos la geometría de polígono por la geometría de punto (centroide)
    # Nota: Aparecerá una advertencia (Warning) si tu CRS es geográfico (grados) en lugar de proyectado (metros). 
    # Para fines generales, el cálculo sigue siendo válido.
    gdf_centroides['geometry'] = gdf_poligonos.centroid
    
    # ==========================================
    # 4. Guardar el resultado
    # ==========================================
    print("Guardando el nuevo archivo GeoPackage...")
    gdf_centroides.to_file(archivo_salida, driver="GPKG")
    
    print("¡Proceso completado con éxito! Revisa tu archivo de salida.")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Leyendo la capa de polígonos...
Calculando los centroides...
Guardando el nuevo archivo GeoPackage...
¡Proceso completado con éxito! Revisa tu archivo de salida.


## MST CON LA CAPA PREDICHA

In [ ]:
import geopandas as gpd
import network as nx 
import numpy as np
from sc

In [ ]:
import geopandas as gpd
import networkx as nx
import numpy as np
from scipy.spatial import distance_matrix
from shapely.geometry import LineString

# ==========================================
# 1. Configuración de rutas y variables
# ==========================================
archivo_entrada = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_mst.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

try:
    # ==========================================
    # 2. Cargar los datos (puntos/centroides)
    # ==========================================
    print("Leyendo la capa de centroides...")
    gdf_puntos = gpd.read_file(archivo_entrada)
    
    # Validar que el atributo exista en la capa
    if atributo_clave not in gdf_puntos.columns:
        raise ValueError(f"El atributo '{atributo_clave}' no existe en las columnas de la capa.")

    # ==========================================
    # 3. Preparar coordenadas y calcular distancias
    # ==========================================
    print("Extrayendo coordenadas y calculando distancias...")
    # Extraemos X e Y de cada punto
    coords = np.array([(geom.x, geom.y) for geom in gdf_puntos.geometry])
    
    # Calculamos la matriz de distancias entre todos los puntos
    dist_matrix = distance_matrix(coords, coords)
    
    # ==========================================
    # 4. Crear el Grafo y calcular el MST
    # ==========================================
    print("Calculando el Árbol de Expansión Mínima (MST)...")
    # Construimos un grafo a partir de la matriz de distancias
    G = nx.from_numpy_array(dist_matrix)
    
    # La magia ocurre aquí: networkx calcula el MST automáticamente
    T = nx.minimum_spanning_tree(G)
    
    # ==========================================
    # 5. Reconstruir la geometría (Líneas) con el Atributo
    # ==========================================
    print("Generando las líneas de la red...")
    lineas = []
    ubigeos_origen = []
    ubigeos_destino = []
    distancias = []
    
    # Recorremos cada "arista" (conexión) del árbol mínimo
    for u, v, data in T.edges(data=True):
        punto_u = gdf_puntos.iloc[u]
        punto_v = gdf_puntos.iloc[v]
        
        # Crear la línea que conecta ambos puntos
        linea = LineString([punto_u.geometry, punto_v.geometry])
        lineas.append(linea)
        
        # Guardar el UBIGEO de cada extremo
        ubigeos_origen.append(punto_u[atributo_clave])
        ubigeos_destino.append(punto_v[atributo_clave])
        
        # Guardar la distancia
        distancias.append(data['weight'])
        
    # ==========================================
    # 6. Exportar resultado a GeoPackage
    # ==========================================
    print("Guardando capa de líneas del MST...")
    # Creamos un nuevo GeoDataFrame para las líneas
    gdf_mst = gpd.GeoDataFrame({
        f"{atributo_clave}_ORIGEN": ubigeos_origen,
        f"{atributo_clave}_DESTINO": ubigeos_destino,
        "DISTANCIA": distancias,
        "geometry": lineas
    }, crs=gdf_puntos.crs) # Mantenemos el mismo Sistema de Coordenadas
    
    # Guardamos a .gpkg
    gdf_mst.to_file(archivo_salida, driver="GPKG")
    print("¡Proceso completado! La red MST ha sido generada.")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Leyendo la capa de centroides...
Extrayendo coordenadas y calculando distancias...
Calculando el Árbol de Expansión Mínima (MST)...


### CON EPSG:32718

In [2]:
import geopandas as gpd
import networkx as nx
import numpy as np
import math
from scipy.spatial import Delaunay
from shapely.geometry import LineString

# ==========================================
# 1. Configuración de rutas y variables
# ==========================================
archivo_entrada = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_mst_rapido.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

try:
    # ==========================================
    # 2. Cargar los datos y Filtrar
    # ==========================================
    print("Leyendo la capa de centroides...")
    gdf_puntos = gpd.read_file(archivo_entrada)
    
    print(f"Filtrando puntos donde {atributo_clave} es 0...")
    gdf_puntos = gdf_puntos[(gdf_puntos[atributo_clave] != 0) & (gdf_puntos[atributo_clave] != '0')]
    gdf_puntos = gdf_puntos.reset_index(drop=True)
    print(f"Puntos restantes para procesar en el MST: {len(gdf_puntos)}")

    # ==========================================
    # 3. EXTRAER COORDENADAS Y APLICAR DELAUNAY (¡EL TRUCO RÁPIDO!)
    # ==========================================
    print("Extrayendo coordenadas y calculando Triangulación de Delaunay...")
    coords = np.array([(geom.x, geom.y) for geom in gdf_puntos.geometry])
    
    # Esto crea una red de triángulos solo entre vecinos cercanos en milisegundos
    tri = Delaunay(coords)
    
    print("Construyendo el grafo optimizado...")
    # Extraemos solo las líneas de esos triángulos para no cruzar todo con todo
    edges = set()
    for simplex in tri.simplices:
        edges.add(tuple(sorted([simplex[0], simplex[1]])))
        edges.add(tuple(sorted([simplex[1], simplex[2]])))
        edges.add(tuple(sorted([simplex[2], simplex[0]])))

    # ==========================================
    # 4. Crear el Grafo y calcular el MST
    # ==========================================
    G = nx.Graph()
    
    # Añadimos las líneas al grafo calculando su distancia real
    for u, v in edges:
        p1 = coords[u]
        p2 = coords[v]
        dist = math.hypot(p1[0] - p2[0], p1[1] - p2[1]) # Distancia euclidiana rápida
        G.add_edge(u, v, weight=dist)

    print("Calculando el Árbol de Expansión Mínima (MST)...")
    # Como ahora hay miles de aristas en vez de millones, esto será instantáneo
    T = nx.minimum_spanning_tree(G)
    
    # ==========================================
    # 5. Reconstruir la geometría (Líneas)
    # ==========================================
    print("Generando las líneas de la red final...")
    lineas = []
    ubigeos_origen = []
    ubigeos_destino = []
    distancias = []
    
    for u, v, data in T.edges(data=True):
        punto_u = gdf_puntos.iloc[u]
        punto_v = gdf_puntos.iloc[v]
        
        linea = LineString([punto_u.geometry, punto_v.geometry])
        lineas.append(linea)
        
        ubigeos_origen.append(punto_u[atributo_clave])
        ubigeos_destino.append(punto_v[atributo_clave])
        distancias.append(data['weight'])
        
    # ==========================================
    # 6. Exportar resultado a GeoPackage
    # ==========================================
    print("Guardando capa de líneas del MST...")
    gdf_mst = gpd.GeoDataFrame({
        f"{atributo_clave}_ORIGEN": ubigeos_origen,
        f"{atributo_clave}_DESTINO": ubigeos_destino,
        "DISTANCIA": distancias,
        "geometry": lineas
    }, crs=gdf_puntos.crs)
    
    gdf_mst.to_file(archivo_salida, driver="GPKG")
    print("¡Proceso completado a la velocidad de la luz!")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Leyendo la capa de centroides...
Filtrando puntos donde UBIGEO_CCPP_CONFIRMADO es 0...
Puntos restantes para procesar en el MST: 26312
Extrayendo coordenadas y calculando Triangulación de Delaunay...
Construyendo el grafo optimizado...
Calculando el Árbol de Expansión Mínima (MST)...
Generando las líneas de la red final...
Guardando capa de líneas del MST...
¡Proceso completado a la velocidad de la luz!


In [3]:
import geopandas as gpd
import networkx as nx
import numpy as np
import math
from scipy.spatial import Delaunay, distance_matrix
from shapely.geometry import LineString

# ==========================================
# 1. Configuración de rutas y variables
# ==========================================
archivo_entrada = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_centroides.gpkg"
archivo_salida = r"C:\Users\decg112\Downloads\capas de cajamarca\cajamarca diego_1\viviendas_regu\borradores\capa_mst_por_ubigeo.gpkg"
atributo_clave = "UBIGEO_CCPP_CONFIRMADO"

try:
    # ==========================================
    # 2. Cargar y Filtrar datos
    # ==========================================
    print("Leyendo la capa de centroides...")
    gdf_puntos = gpd.read_file(archivo_entrada)
    
    print(f"Filtrando puntos donde {atributo_clave} es 0...")
    gdf_puntos = gdf_puntos[(gdf_puntos[atributo_clave] != 0) & (gdf_puntos[atributo_clave] != '0')]
    
    # ==========================================
    # 3. AGRUPAR POR UBIGEO
    # ==========================================
    grupos_ubigeo = gdf_puntos.groupby(atributo_clave)
    
    lineas_totales = []
    ubigeos_totales = []
    distancias_totales = []
    
    print(f"Se encontraron {len(grupos_ubigeo)} UBIGEOs distintos.")
    print("Calculando redes MST locales... (esto tomará unos segundos)")

    # Iteramos sobre cada UBIGEO de forma independiente
    for ubigeo, grupo_df in grupos_ubigeo:
        n_puntos = len(grupo_df)
        
        # Si un UBIGEO solo tiene 1 vivienda/punto, no hay nada que conectar
        if n_puntos < 2:
            continue
            
        # Resetear índice localmente es vital
        grupo_df = grupo_df.reset_index(drop=True)
        coords = np.array([(geom.x, geom.y) for geom in grupo_df.geometry])
        
        G = nx.Graph()
        
        # ==========================================
        # 4. Lógica Híbrida (Delaunay vs Matriz)
        # ==========================================
        # Delaunay necesita mínimo 4 puntos para no fallar. 
        if n_puntos < 4:
            # Para 2 o 3 puntos, la matriz clásica es instantánea y segura
            dist_mat = distance_matrix(coords, coords)
            G = nx.from_numpy_array(dist_mat)
        else:
            try:
                # Para grupos grandes, usamos Delaunay
                tri = Delaunay(coords)
                edges = set()
                for simplex in tri.simplices:
                    edges.add(tuple(sorted([simplex[0], simplex[1]])))
                    edges.add(tuple(sorted([simplex[1], simplex[2]])))
                    edges.add(tuple(sorted([simplex[2], simplex[0]])))
                    
                for u, v in edges:
                    p1 = coords[u]
                    p2 = coords[v]
                    dist = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
                    G.add_edge(u, v, weight=dist)
            except Exception:
                # Respaldo: Si todos los puntos de un UBIGEO están en una línea recta perfecta, 
                # Delaunay falla. Usamos la matriz clásica como salvavidas.
                dist_mat = distance_matrix(coords, coords)
                G = nx.from_numpy_array(dist_mat)
                
        # ==========================================
        # 5. Calcular MST y guardar líneas de este UBIGEO
        # ==========================================
        T = nx.minimum_spanning_tree(G)
        
        for u, v, data in T.edges(data=True):
            punto_u = grupo_df.iloc[u]
            punto_v = grupo_df.iloc[v]
            
            linea = LineString([punto_u.geometry, punto_v.geometry])
            lineas_totales.append(linea)
            
            # Solo guardamos el UBIGEO una vez, ya que ambos puntos pertenecen al mismo
            ubigeos_totales.append(ubigeo)
            distancias_totales.append(data['weight'])

    # ==========================================
    # 6. Exportar resultado a GeoPackage
    # ==========================================
    print("Guardando capa de líneas...")
    gdf_mst = gpd.GeoDataFrame({
        atributo_clave: ubigeos_totales,
        "DISTANCIA_M": distancias_totales,
        "geometry": lineas_totales
    }, crs=gdf_puntos.crs)
    
    gdf_mst.to_file(archivo_salida, driver="GPKG")
    print(f"¡Éxito! Se generaron {len(lineas_totales)} líneas en total divididas por UBIGEO.")

except Exception as e:
    print(f"Ocurrió un error: {e}")

Leyendo la capa de centroides...
Filtrando puntos donde UBIGEO_CCPP_CONFIRMADO es 0...
Se encontraron 104 UBIGEOs distintos.
Calculando redes MST locales... (esto tomará unos segundos)
Guardando capa de líneas...
¡Éxito! Se generaron 26208 líneas en total divididas por UBIGEO.
